# 09. SFT Runbook and Masking

- Parent issue: `#25`
- Status: `active`
- Summary: Define the adapter training runbook, masking policy, and checkpoint rules before the first training execution.

## Audience and Why It Matters

Training owners, reviewers, and stakeholders who need a traceable explanation of what the adapter is learning and why.

## Decision / Hypothesis

Start with a conservative LoRA/QLoRA plan, keep target modules configurable, and make loss masking and checkpointing explicit artifacts rather than tacit trainer settings.

## Environment and Reproduction

This notebook executes cleanly before the full model weights exist - masking walkthroughs use toy token IDs only.

- Python: 3.11+
- Run from repo root or open directly in Jupyter
- Expected companion issue and registry entry: `docs/execution/NOTEBOOKS.md`

In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")

In [ ]:
# Device detection: prefers CUDA, falls back to CPU gracefully.
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir() and (_candidate / 'notebooks').is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.device import log_device_info
device = log_device_info()

## Method and Outputs

### Planned method
- Document the planned training stack and hardware modes.
- Describe the masking policy and why it matters.
- Define checkpoint, regression, and provenance requirements.

### Expected outputs
- Runbook outline
- Masking decision log
- Training risk register

In [ ]:
planned_tasks = [
    "replace scaffold text with verified findings",
    "link concrete artifacts produced during execution",
    "update docs/execution/NOTEBOOKS.md when status changes",
]

for item in planned_tasks:
    print(f"- {item}")

## Section 1 - Loss Masking Walkthrough

In supervised fine-tuning, **loss masking** ensures the model only learns to predict
assistant tokens - not the system prompt or user turns given as input.

### How `apply_loss_mask` works

`src/training/sft_trainer.py::apply_loss_mask` scans the token sequence for role
boundaries using `tokenizer.role_start_ids`. Any position belonging to a
**non-assistant** turn is set to `IGNORE_INDEX = -100`. PyTorch cross-entropy
loss skips positions with label `-100`, so the gradient is computed only over the
assistant response tokens.

**Key invariant:** masking is applied **once** by `tokenize_dataset.py` and stored
in the `.pt` shard files. `run_sft.py` reads pre-masked labels via
`PassthroughCollator` and must **not** call `apply_loss_mask` again.

Run the cell below to see this in action with toy token IDs - no model download needed.

In [ ]:
# Masking walkthrough -- runs offline; no weights required
import sys
from pathlib import Path

_root = Path.cwd()
for _p in [_root] + list(_root.parents):
    if (_p / 'src').is_dir():
        _root = _p
        break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.contracts import SFTExample
from src.training.sft_trainer import apply_loss_mask, IGNORE_INDEX

# -- 1. Construct a minimal SFTExample -----------------------------------
example = SFTExample(
    example_id="walkthrough-001",
    category="algebra",
    messages=[
        {"role": "user",      "content": "What is 2 + 2?"},
        {"role": "assistant", "content": "\\boxed{4}"},
    ],
    completion="\\boxed{4}",
    source="synthetic",
    split="train",
)

print("SFTExample messages:")
for msg in example.messages:
    print(f"  [{msg['role']:>9}]  {msg['content']!r}")
print()

# -- 2. Mock tokenizer: stands in for AutoTokenizer ----------------------
# In production tokenize_dataset.py builds role_start_ids from the chat
# template. We hard-code plausible small integers here.
class _WalkthroughTokenizer:
    """Minimal stand-in exposing role_start_ids used by apply_loss_mask."""
    role_start_ids: dict = {
        "user":      [1, 2],   # opening tokens of a user turn
        "assistant": [3, 4],   # opening tokens of an assistant turn
    }
    def encode(self, text: str) -> list:
        return [ord(c) % 128 for c in text[:8]]

tokenizer = _WalkthroughTokenizer()

# -- 3. Build a toy token sequence ---------------------------------------
# Layout: <user_start> [user body] <asst_start> [asst body]
input_ids = (
    [1, 2]           # <user> marker   (role_start_ids["user"])
    + [10, 11, 12]   # 'What is 2+2?' body tokens (mock)
    + [3, 4]         # <assistant> marker  (role_start_ids["assistant"])
    + [20, 21, 22]   # '\\boxed{4}' body tokens (mock)
)
labels_before = list(input_ids)

# -- 4. Apply the loss mask ----------------------------------------------
masked_labels = apply_loss_mask(
    input_ids=input_ids,
    labels=labels_before,
    tokenizer=tokenizer,
)

# -- 5. Visualise kept vs masked positions --------------------------------
print(f"{'pos':>4}  {'token_id':>9}  {'lbl_before':>10}  {'lbl_after':>9}  status")
print("-" * 72)
for pos, (tok, lbl) in enumerate(zip(input_ids, masked_labels)):
    status = "kept  <- loss computed" if lbl != IGNORE_INDEX else "MASKED  (-100, loss skipped)"
    print(f"{pos:>4}  {tok:>9}  {labels_before[pos]:>10}  {lbl:>9}  {status}")

print()
n_kept   = sum(1 for l in masked_labels if l != IGNORE_INDEX)
n_masked = sum(1 for l in masked_labels if l == IGNORE_INDEX)
print(f"Kept positions (assistant body) : {n_kept}")
print(f"Masked positions (user + marker): {n_masked}")


## Section 2 - Full Pipeline Diagram

Each stage gates the next. Stages 4-6 run post-training.

```
scripts/hpc/preflight.sh
    |  checks: GPU presence, VRAM >= threshold, disk space, config validity
    |  flags : --local (skip GPU check for smoke tests)
    |          --write-config (persist merged config to checkpoint dir)
    v
scripts/hpc/tokenize_dataset.py
    |  input : JSONL  (SFTExample rows)
    |  output: .pt shard files  +  dataset_fingerprint.json
    |  NOTE  : calls apply_loss_mask once - labels pre-masked in shards
    v
scripts/hpc/run_sft.py
    |  input : .pt shards  +  configs/lora_baseline.yaml (or child via extends:)
    |  output: step-XXXXX/ checkpoints inside --output-dir
    |  loads base model, wraps with LoRA (PEFT), runs TRL SFTTrainer
    |  uses PassthroughCollator - NO re-masking
    v
scripts/hpc/checkpoint_policy.py
    |  input : checkpoint dir
    |  output: pruned dir (retains N most recent step-XXXXX/ subdirs)
    |  called automatically by run_sft.py at end of training
    v
scripts/hpc/regression_gate.py
    |  input : best checkpoint  +  data/golden_20.jsonl (20 held-out examples)
    |  output: exit 0 (PASS) or exit 1 (FAIL)
    |  MUST exit 0 before package_adapter.py is allowed to run
    v
scripts/hpc/package_adapter.py
    |  input : checkpoint that cleared the gate
    |  output: submissions/submission.zip
    |  zip contents: adapter_config.json + adapter_model.safetensors at root
    v
   submission.zip  ->  Kaggle submission
```

**Config inheritance:** `configs/lora_baseline.yaml` is the base config.
Child configs extend it via `extends: configs/lora_baseline.yaml`.
`run_sft.py::load_config` resolves the full chain with deep-merge (child wins).

## Section 3 - Reference CLI Invocations

The cells below show the exact commands that drive the pipeline.
**Do not execute training from this notebook** - use the cells as a copy-paste reference.

In [ ]:
# Reference CLI invocations -- documentation only, do NOT run training

# -- 3a. Local / interactive run (smoke test, CPU or single-GPU) ---------
print("=== 3a. Local training run ===")
print(
    "python scripts/hpc/run_sft.py \\\n"
    "    --config   configs/lora_baseline.yaml \\\n"
    "    --data-dir /path/to/tokenized_shards \\\n"
    "    --output-dir /path/to/run_output"
)
print()

# -- 3b. SLURM cluster submission via submit_sft.sbatch ------------------
#     The sbatch script orchestrates all stages:
#     preflight -> tokenize_dataset -> run_sft -> checkpoint_policy
print("=== 3b. SLURM cluster submission ===")
print(
    "sbatch scripts/hpc/submit_sft.sbatch \\\n"
    "    configs/lora_baseline.yaml"
)
print()

# -- Key settings in configs/lora_baseline.yaml -------------------------
print("=== Key settings in configs/lora_baseline.yaml ===")
settings = [
    ("base_model",        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"),
    ("lora_r",            "32  <- evaluator cap is r <= 32"),
    ("lora_alpha",        "64"),
    ("lora_dropout",      "0.05"),
    ("target_modules",    "in_proj, out_proj, up_proj, down_proj"),
    ("load_in_4bit",      "false  (set true for QLoRA / lower VRAM)"),
    ("torch_dtype",       "bfloat16"),
    ("trust_remote_code", "true"),
]
for k, v in settings:
    print(f"  {k:<22}: {v}")


## Section 4 - Resume Semantics

Training can be interrupted (SLURM preemption, OOM, manual cancel) and resumed
from the latest saved checkpoint. Two mechanisms are available:

- **Explicit path:** resolve with `resume_from_latest.py`, pass the result directly.
- **Auto-detect:** pass `--resume-from-checkpoint true`; HF Trainer scans `output-dir`.

In [ ]:
# Resume semantics reference -- shows commands, does not execute training

# -- 4a. Shell: resolve the latest step-XXXXX checkpoint path ------------
# resume_from_latest.py scans for subdirs matching step-\d+ and returns
# the one with the highest step number (absolute path printed to stdout).
print("=== 4a. Shell - resolve latest checkpoint ===")
print(
    "LATEST=$(python scripts/hpc/resume_from_latest.py \\\n"
    "    --checkpoint-dir /path/to/run_output)\n"
    "echo \"Resuming from: $LATEST\""
)
print()

# -- 4b. run_sft.py with explicit resume path ----------------------------
print("=== 4b. run_sft.py with explicit resume path ===")
print(
    "python scripts/hpc/run_sft.py \\\n"
    "    --config                  configs/lora_baseline.yaml \\\n"
    "    --data-dir                /path/to/tokenized_shards \\\n"
    "    --output-dir              /path/to/run_output \\\n"
    "    --resume-from-checkpoint  /path/to/run_output/step-00500"
)
print()

# -- 4c. Shorthand: pass 'true' to let HF Trainer auto-detect -----------
print("=== 4c. run_sft.py with Trainer auto-detect ('true') ===")
print(
    "python scripts/hpc/run_sft.py \\\n"
    "    --config                  configs/lora_baseline.yaml \\\n"
    "    --data-dir                /path/to/tokenized_shards \\\n"
    "    --output-dir              /path/to/run_output \\\n"
    "    --resume-from-checkpoint  true"
)
print()

# -- 4d. How submit_sft.sbatch wires auto-resume (lines 123-133) --------
print("=== 4d. submit_sft.sbatch auto-resume snippet ===")
print(
    'RESUME_ARG=""\n'
    'if LATEST=$(python scripts/hpc/resume_from_latest.py "$RUN_DIR/checkpoints" 2>/dev/null); then\n'
    '    [ -n "$LATEST" ] && RESUME_ARG="--resume-from-checkpoint $LATEST"\n'
    'fi\n'
    '\n'
    "python scripts/hpc/run_sft.py \\\n"
    '    --config     "$CONFIG_PATH" \\\n'
    '    --data-dir   "$TOKENIZED_DIR" \\\n'
    '    --output-dir "$RUN_DIR" \\\n'
    '    $RESUME_ARG'
)


## Section 5 - Promotion Criteria: Regression Gate Before Packaging

A checkpoint is **only eligible for submission packaging** if it first passes the
regression gate against the golden set.

### Gate command

```bash
python scripts/hpc/regression_gate.py \\
    --checkpoint /path/to/run_output/step-XXXXX \\
    --golden-set data/golden_20.jsonl
```

### Gate logic

| Step | Detail |
|------|--------|
| Evaluate  | Run checkpoint on `data/golden_20.jsonl` (20 held-out examples) |
| Extract   | Parse outputs with `\\boxed{}` extraction + `1e-3` numeric tolerance |
| Threshold | Compare accuracy against stored baseline score |
| Exit 0    | **PASS** - checkpoint cleared; packaging is allowed |
| Exit 1    | **FAIL** - checkpoint regressed; packaging is blocked |

### Packaging - only after the gate passes

```bash
python scripts/hpc/package_adapter.py \\
    --checkpoint /path/to/run_output/step-XXXXX \\
    --output    submissions/submission.zip
```

The resulting zip must contain **exactly two files at the root**:

```
submission.zip
├── adapter_config.json
└── adapter_model.safetensors
```

> **Never bypass `regression_gate.py`.** Submitting a regressed adapter wastes a
> submission slot and risks a lower leaderboard score. The gate is the final
> quality checkpoint before the adapter leaves the training environment.

## Results / Open Risks

This section will be updated as training runs complete.

- Competition constraints may invalidate aggressive LoRA settings.
- Poor masking decisions can waste gradient budget or overfit formatting.
- Checkpoint rotation policy must keep enough history to allow gate-based rollback.

## Sources

- [Execution plan v0.2](docs/planning/plan_v0.2.md)
- [PEFT LoRA docs](https://huggingface.co/docs/peft/en/package_reference/lora)
- [TRL docs](https://huggingface.co/docs/trl/main/en/index)
- `src/training/sft_trainer.py` - `apply_loss_mask` implementation
- `scripts/hpc/run_sft.py` - SFT training launcher
- `scripts/hpc/resume_from_latest.py` - checkpoint resolver
- `scripts/hpc/submit_sft.sbatch` - SLURM job script